# 00b_download_structures.py

Download the verified DSSTox structure release used to enrich ToxCast identifiers.

In [ ]:
from pathlib import Path
import os, sys, runpy
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
print('Project root:', PROJECT_ROOT)

## 2. Implementation

Every function below is copied verbatim from `scripts/00b_download_structures`, in source order, kept in sync by `python scripts/sync_notebooks.py` (and checked by `11_validate_project.py`). Edit the `.py` file, then re-run the sync script -- never hand-edit these cells.

In [ ]:
"""STEP 0b -- Download EPA's DSSTox chemical-structure dump (DTXSID -> SMILES,
name, CASRN). The ToxCast MySQL dump has chemical identifiers but no SMILES;
this file is what 01b_prepare_structures.py joins in to fill that gap.
File size is verified against a known-good value so a truncated/interrupted
download is never silently treated as complete."""
import json
import zipfile
import requests
from core.paths import DATA_SOURCE

FILE_ID='69529775e4b0731a616efc4b'
NAME='DSSTox_CCD_dump_12092025_CSVs.zip'
SIZE=289824966


#### `main`

Download the DSSTox zip if it's missing or the wrong size, then list its

In [ ]:
def main():
    """Download the DSSTox zip if it's missing or the wrong size, then list its
    contents and record provenance (source URL, release, size) alongside it."""
    destination=DATA_SOURCE/'dsstox'/NAME
    destination.parent.mkdir(parents=True,exist_ok=True)
    url=f'https://clowder.edap-cluster.com/api/files/{FILE_ID}/blob'
    if not destination.exists() or destination.stat().st_size!=SIZE:
        part=destination.with_suffix('.zip.part')
        with requests.get(url,stream=True,timeout=(30,300)) as response:
            response.raise_for_status()
            with part.open('wb') as out:
                for chunk in response.iter_content(8*1024*1024): out.write(chunk)
        if part.stat().st_size!=SIZE: raise IOError('Incomplete DSSTox download; rerun.')
        part.replace(destination)
    with zipfile.ZipFile(destination) as archive:
        print('Downloaded:',destination)
        for item in archive.infolist(): print(item.filename,item.file_size)
    destination.with_suffix('.provenance.json').write_text(json.dumps({'url':url,'release':'December 2025','size':SIZE},indent=2))


In [ ]:
# Set RUN_STEP=True when you are ready to execute this workflow.
# The notebook defaults to False so "Run All" is safe and does not accidentally
# download large files, start a long training job, or overwrite project outputs.
RUN_STEP = False

# Command-line arguments used when RUN_STEP=True.
RUN_ARGS = []

if RUN_STEP:
    old = sys.argv[:]
    try:
        sys.argv = ['00b_download_structures.py'] + RUN_ARGS
        try:
            main()
        except SystemExit as exc:
            # main() uses SystemExit(0) as a CLI success signal (e.g. --check).
            # A terminal treats that as silent success; Jupyter displays *any*
            # SystemExit as an error-looking traceback, so only re-raise on an
            # actual failure (nonzero/non-None exit code).
            if exc.code not in (0, None):
                raise
    finally:
        sys.argv = old
else:
    print('Implementation loaded successfully.')
    print("Set RUN_STEP = True in this cell to download structures")
    print('RUN_ARGS =', RUN_ARGS)
